# 5-1. **Topic Modeling**

In [1]:
!pip install -q nltk
!pip install -q konlpy
!pip install -q gensim

In [2]:
import pandas as pd

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
!wget https://raw.githubusercontent.com/kimtwan/NLP_lecture/master/data/MPB_minutes.tsv

--2026-05-10 02:29:52--  https://raw.githubusercontent.com/kimtwan/NLP_lecture/master/data/MPB_minutes.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15195720 (14M) [application/octet-stream]
Saving to: ‘MPB_minutes.tsv.1’

MPB_minutes.tsv.1   100%[===================>]  14.49M  76.9MB/s    in 0.2s    

2026-05-10 02:29:52 (76.9 MB/s) - ‘MPB_minutes.tsv.1’ saved [15195720/15195720]



In [ ]:
# load Monetary Policy Board minutes text
data = pd.read_csv('MPB_minutes.tsv', encoding='utf-8', sep='\t', parse_dates=['날짜'], index_col='날짜')
# minutes after 2020
data = data.sort_index().loc['2020-01-01':]
data.head()

In [5]:
from konlpy.tag import Kkma
from nltk import sent_tokenize
from tqdm import notebook

In [6]:
kkma = Kkma()
corpus = data['의사록'].tolist()

corpus_nouns = []
# takes aboute 9 mins
for i in notebook.tqdm(range(len(corpus))):
    # separate the minutes into sentences
    sentences = sent_tokenize(corpus[i])
    for sentence in sentences:
        # extract nouns
        nouns = kkma.nouns(sentence)
        # includes only words longer than 2 characters
        nouns = [noun for noun in nouns if len(noun) > 1]
        corpus_nouns.append(nouns)

  0%|          | 0/23 [00:00<?, ?it/s]

In [7]:
import pprint
pprint.pprint(corpus_nouns[:5])

[['2020', '2020년', '1차', '금융', '금융통화위원회', '통화', '위원회', '의사록'],
 ['2020', '2020년', '1월', '17', '17일'],
 ['금융', '금융통화위원회', '통화', '위원회', '회의실'],
 ['출석', '출석위원', '위원'],
 ['결석', '결석위원', '위원']]


In [8]:
# topic analysis using LDA
from gensim.test.utils import common_texts
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel

In [9]:
# convert sentences into a form that can be used with gensim LDA
dictionary = Dictionary(corpus_nouns)
corpus = [dictionary.doc2bow(text) for text in corpus_nouns]

In [10]:
pprint.pprint(corpus[:5])

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1)],
 [(1, 1), (2, 1), (8, 1), (9, 1), (10, 1)],
 [(3, 1), (4, 1), (5, 1), (7, 1), (11, 1)],
 [(12, 1), (13, 1), (14, 1)],
 [(12, 1), (15, 1), (16, 1)]]


In [11]:
# set number of topics
num_topics = 5

# run the model
lda = LdaModel(corpus, num_topics=num_topics, id2word=dictionary)

In [12]:
# print topics
topics = lda.print_topics(num_words=10)
for topic in topics:
    print(topic)

(0, '0.026*"관련" + 0.023*"부서" + 0.023*"관련부서" + 0.021*"물가" + 0.017*"위원" + 0.013*"답변" + 0.013*"언급" + 0.012*"전망" + 0.010*"상승" + 0.009*"가격"')
(1, '0.020*"경제" + 0.012*"소비" + 0.011*"회복" + 0.011*"코로나" + 0.010*"상황" + 0.010*"흐름" + 0.010*"영향" + 0.010*"확산" + 0.009*"있음" + 0.009*"지속"')
(2, '0.022*"금리" + 0.020*"상승" + 0.015*"인상" + 0.012*"물가" + 0.011*"기대" + 0.010*"정책" + 0.009*"시장" + 0.009*"하락" + 0.009*"인플레이션" + 0.009*"통화"')
(3, '0.017*"금융" + 0.016*"대출" + 0.016*"자금" + 0.014*"증가" + 0.010*"시장" + 0.010*"확대" + 0.009*"은행" + 0.009*"기업" + 0.008*"외환" + 0.008*"규모"')
(4, '0.022*"금리" + 0.019*"정책" + 0.015*"통화정책" + 0.015*"통화" + 0.014*"필요" + 0.013*"위원" + 0.012*"가계" + 0.012*"금융" + 0.012*"기준" + 0.011*"기준금리"')
